In [0]:
%run ./00_setup

In [0]:
# Generate Orders_Item data
import pytz
from datetime import datetime

import uuid
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType
from pyspark.sql import functions as F

orders_df = spark.table("workspace.amazon_fullfilment_silver.orders_silver").select("order_id").filter("status='Initial' AND is_current=true").alias("orders")
products_df = spark.table("workspace.amazon_fullfilment_silver.products_silver").select("product_id").filter("is_current=true").alias("products")


tz = pytz.timezone('America/New_York')

num_orders =orders_df.count()

def get_rand_num_order_items():
    if num_orders == 0:
        return 0
    else:
        return random.randint(num_orders,num_orders*2)


num_order_items = get_rand_num_order_items()
def generate_order_items_data(num_order_items=0):
# 2. Create a base of random numbers and join
    # This creates a "random sample" across your available orders and products
    return orders_df.crossJoin(products_df) \
        .orderBy(F.rand()) \
        .limit(num_order_items) \
        .withColumn("order_item_id", F.expr("uuid()")) \
        .withColumn("quantity", (F.rand() * 10 + 1).cast("int")) \
        .select(
            F.col("orders.order_id"), 
            F.col("order_item_id"), 
            F.col("products.product_id"), 
            F.col("quantity"))


# Generate and view
order_items_df = generate_order_items_data(num_order_items)

# save to the source volume

(order_items_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/order_items"))